# MELD cue-only SFT — v2

Measure whether Qwen2.5-Omni can learn the five acoustic cues saved by
`data_processing.ipynb`. This retains the original audio/question → answer SFT,
4-bit NF4 QLoRA, assistant-only loss, and deterministic generation evaluation.

Use **all saved clean training rows** and **the saved clean MELD dev → val split**.
No dynamic split, emotion balancing, six-speaker restriction, feature extraction,
quartile refitting, or generated reasoning. Labels are the saved gender-relative
quartile targets, including the full timbre labels. These diagnostics measure
agreement with that acoustic labeling scheme, not emotion recognition accuracy.

Run top to bottom on a CUDA GPU. Select one cue for an isolated experiment, or
several for shared-adapter multi-task training (one question per audio/cue).
Restart the kernel and use a new run name for each independent experiment.
If GRPO must use disjoint training audio, reserve that partition upstream before
running this notebook; this diagnostic uses the complete saved training manifest.

## 1. Environment and experiment

The permanent preprocessing export lives at
`/content/drive/MyDrive/AGORA_SER/data/meld_16k_mono`.
Before training, copy `meld_16k_mono_clean.tar` from Drive into the Colab runtime
and extract cleaned audio locally under `/content/agora_work/meld_16k_mono`.
If the archive is missing, create it once on Drive from the saved
`emo_tag_clean` directory (including metadata). This first packaging step can be
slow; subsequent sessions transfer one archive instead of reading thousands of
WAVs individually from Drive. If neither the archive nor cleaned source WAVs
exist on Drive, restore the preprocessing export first.

Training and evaluation read local WAVs and manifests. Checkpoints and reports
remain on Drive. A fresh Colab runtime restores the data again; its temporary
storage is only a working copy. Re-running the staging cell refreshes that copy.


In [ ]:
# Optional setup (uncomment in a fresh training runtime):
%pip install -q "transformers==5.20.2" peft accelerate bitsandbytes soundfile pandas scikit-learn

from pathlib import Path
import hashlib
import json
import re
import tarfile
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from IPython.display import display

IS_COLAB = Path('/content').exists()
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/AGORA_SER/data/meld_16k_mono')
DATA_DIR = (Path('/content/agora_work/meld_16k_mono') if IS_COLAB
            else Path.cwd() / 'data/meld_16k_mono')
CLEAN_DIR = DATA_DIR / 'emo_tag_clean'
TRAIN_CSV = CLEAN_DIR / 'clean_train_set_with_cues.csv'
VAL_CSV = CLEAN_DIR / 'clean_val_set_with_cues.csv'
QUARTILES_JSON = CLEAN_DIR / 'gender_acoustic_quartiles.json'

CUES = ('pitch_level', 'pitch_variation', 'vocal_energy', 'speaking_rate', 'timbre')
SELECTED_CUES = CUES  # e.g. ('pitch_level',) for an isolated cue experiment
RUN_NAME = 'all_cues_gender_seed42'
RUN_ROOT = (Path('/content/drive/MyDrive/AGORA_SER/runs/cue_learnability') if IS_COLAB
            else Path.cwd() / 'runs/cue_learnability')
OUTPUT_DIR = RUN_ROOT / RUN_NAME
MODEL_ID = 'Qwen/Qwen2.5-Omni-7B'
SEED = 42
EPOCHS = 1
EVAL_BATCH_SIZE = 2
EVALUATE_BEFORE = True

## 2. Read the preprocessing contract

Paths from Drive or old runtimes are rebased to the local `DATA_DIR`, preserving `emo_tag_clean/train`
and `emo_tag_clean/val`. Missing files, duplicate audio, and split overlap fail
before model loading. Missing cue targets remain missing: validation reports
coverage per cue; training requires complete labels. No rows are silently dropped.

In [ ]:
CUE_LABELS = {cue: ('low', 'moderate', 'high') for cue in CUES}
CUE_LABELS['timbre'] = ('low (muffled/soft)', 'moderate (modal)', 'high (tense/bright)')
CUE_DESCRIPTIONS = {
    'pitch_level': 'median pitch level',
    'pitch_variation': 'pitch variation (pitch range)',
    'vocal_energy': 'vocal energy (perceived loudness)',
    'speaking_rate': 'speaking rate (voiced segments per second)',
    'timbre': 'timbre (voiced spectral balance)',
}
if not SELECTED_CUES or len(set(SELECTED_CUES)) != len(SELECTED_CUES) or set(SELECTED_CUES) - set(CUES):
    raise ValueError('SELECTED_CUES must contain unique known cue names.')


def stage_clean_data(drive_dir, local_dir):
    """Persist a missing archive on Drive, then transfer/extract a local working copy."""
    import shutil
    drive_dir, local_dir = Path(drive_dir), Path(local_dir)
    if local_dir.resolve().is_relative_to(drive_dir.resolve()):
        raise ValueError('Local working directory must be outside Drive data storage')
    metadata = ('clean_train_set_with_cues.csv', 'clean_val_set_with_cues.csv',
                'gender_acoustic_quartiles.json')
    archive = drive_dir / 'meld_16k_mono_clean.tar'
    clean_source = drive_dir / 'emo_tag_clean'
    if not archive.is_file():
        # Validate before publishing an archive; a vanished runtime cannot be recovered from CSVs alone.
        for name in metadata:
            if not (clean_source / name).is_file():
                raise FileNotFoundError(f'Missing Drive preprocessing export: {clean_source / name}')
        for split in ('train', 'val'):
            frame = pd.read_csv(clean_source / f'clean_{split}_set_with_cues.csv')
            if frame.empty:
                raise ValueError(f'Cannot archive an empty {split} manifest')
            for path in frame.audio_path:
                resolve_audio_path(path, split, drive_dir)
        print(f'Creating permanent clean archive on Drive: {archive}', flush=True)
        partial = archive.with_suffix('.tar.partial')
        try:
            with tarfile.open(partial, 'w') as bundle:
                bundle.add(clean_source, arcname='meld_16k_mono/emo_tag_clean')
            partial.replace(archive)
        finally:
            partial.unlink(missing_ok=True)

    local_dir.mkdir(parents=True, exist_ok=True)
    local_archive = local_dir.parent / archive.name
    print(f'Copying Drive archive to local runtime: {local_archive}', flush=True)
    partial = local_archive.with_suffix('.tar.partial')
    try:
        shutil.copyfile(archive, partial)
        partial.replace(local_archive)
    finally:
        partial.unlink(missing_ok=True)

    print(f'Extracting cleaned audio locally: {local_dir}', flush=True)
    restored = 0
    extracted_audio = set()
    with tarfile.open(local_archive, 'r:') as bundle:
        for member in bundle:
            parts = Path(member.name).parts
            # Support preprocessing's whole-data archive while excluding raw audio and archive links.
            if (not member.isfile() or '..' in parts or len(parts) not in (3, 4)
                    or parts[:2] != ('meld_16k_mono', 'emo_tag_clean')):
                continue
            is_audio = len(parts) == 4 and parts[2] in ('train', 'val', 'test') and Path(parts[3]).suffix == '.wav'
            is_metadata = len(parts) == 3 and Path(parts[2]).suffix in ('.csv', '.json')
            if not (is_audio or is_metadata):
                continue
            target = local_dir.joinpath(*parts[1:])
            if not target.resolve().is_relative_to(local_dir.resolve()):
                raise ValueError(f'Archive target escapes local working directory: {target}')
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(target.suffix + '.partial')
            try:
                with bundle.extractfile(member) as source, partial.open('wb') as destination:
                    shutil.copyfileobj(source, destination)
                partial.replace(target)
            finally:
                partial.unlink(missing_ok=True)
            if is_audio:
                extracted_audio.add(str(target.resolve()))
            restored += 1
    # Prefer the separately saved Drive manifests when available (they may be newer than the tar).
    for name in metadata:
        source = clean_source / name
        target = local_dir / 'emo_tag_clean' / name
        if source.is_file():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
        if not target.is_file():
            raise FileNotFoundError(f'Missing metadata in Drive export/archive: {name}')
    for split in ('train', 'val'):
        frame = pd.read_csv(local_dir / 'emo_tag_clean' / f'clean_{split}_set_with_cues.csv')
        for path in frame.audio_path:
            resolved = resolve_audio_path(path, split, local_dir)
            if resolved not in extracted_audio:
                raise ValueError(f'Archive is missing a manifest audio; refresh the Drive archive: {resolved}')
    print(f'Staged {restored:,} files. Training audio root: {local_dir}')


def resolve_audio_path(value, split, data_dir):
    if pd.isna(value):
        raise ValueError('Missing audio_path')
    path = Path(str(value))
    parts = path.parts
    if 'emo_tag_clean' in parts:
        relative = Path(*parts[parts.index('emo_tag_clean') + 1:])
    elif not path.is_absolute():
        relative = path
    else:
        raise ValueError(f'Expected a cleaned audio path: {path}')
    if not relative.parts or relative.parts[0] != split or '..' in relative.parts:
        raise ValueError(f'Unexpected {split} path: {path}')
    resolved = (data_dir / 'emo_tag_clean' / relative).resolve()
    if not resolved.is_relative_to((data_dir / 'emo_tag_clean' / split).resolve()):
        raise ValueError(f'Audio escapes split directory: {path}')
    if not resolved.is_file():
        raise FileNotFoundError(resolved)
    return str(resolved)


def load_manifest(csv_path, split, data_dir):
    df = pd.read_csv(csv_path)
    required = {'audio_path', 'effective_gender'} | {f'{cue}_cue' for cue in CUES}
    if required - set(df):
        raise ValueError(f'{csv_path}: missing columns {required - set(df)}')
    if df.empty:
        raise ValueError(f'{csv_path}: empty split')
    if not df.effective_gender.isin(['F', 'M', '__GLOBAL__']).all():
        raise ValueError(f'{csv_path}: missing or invalid effective_gender; rerun preprocessing')
    df['audio_path'] = df.audio_path.map(lambda p: resolve_audio_path(p, split, data_dir))
    if df.audio_path.duplicated().any():
        raise ValueError(f'{csv_path}: duplicate audio')
    for cue in CUES:
        labels = df[f'{cue}_cue']
        unknown = set(labels.dropna()) - set(CUE_LABELS[cue])
        if unknown:
            raise ValueError(f'{csv_path}: unknown {cue} labels {unknown}')
        if split == 'train' and labels.isna().any():
            raise ValueError(f'{csv_path}: missing training labels for {cue}; check preprocessing')
    return df


def check_disjoint(train, val):
    if set(train.audio_path) & set(val.audio_path):
        raise ValueError('Training and validation audio overlap')


if IS_COLAB:
    stage_clean_data(DRIVE_DATA_DIR, DATA_DIR)

train_set = load_manifest(TRAIN_CSV, 'train', DATA_DIR)

stratify_columns = [
    'pitch_level_cue',
    'pitch_variation_cue',
    'vocal_energy_cue',
    'speaking_rate_cue',
    'timbre_cue',
    'effective_gender',
]

strata = train_set[stratify_columns].astype(str).agg('_'.join, axis=1)

counts = strata.value_counts()
strata = strata.where(strata.isin(counts[counts >= 2].index), 'rare')

train_set, _ = train_test_split(
    train_set,
    train_size=0.2,
    stratify=strata,
    random_state=42,
)


val_set = load_manifest(VAL_CSV, 'val', DATA_DIR)
check_disjoint(train_set, val_set)
# Provenance only: labels above are authoritative and are never recomputed here.
quartiles = json.loads(QUARTILES_JSON.read_text())
print(f'Saved MELD splits: train={len(train_set):,}, val={len(val_set):,}')
display(pd.DataFrame({cue: val_set[f'{cue}_cue'].value_counts(dropna=False) for cue in CUES}).fillna(0))

In [11]:
display(train_set)

,audio_path,time,utterance,dialogue,emotion,text,speaker,duration,gender,source_audio_path,...,pitch_variation,vocal_energy,speaking_rate,timbre,effective_gender,pitch_level_cue,pitch_variation_cue,vocal_energy_cue,speaking_rate_cue,timbre_cue
1291,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,4.181,8,222,joy,"So the divorce, the marriage, weve got a lot ...",Doug,2.870000,M,/content/prep_meld/preprocessed_train_splits/d...,...,5.727894,1.097572,2.135231,-5.906699,M,high,moderate,moderate,low,high (tense/bright)
2713,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,2.603,3,480,neutral,"Oh hey, right back at ya.",Ross,1.639000,M,/content/prep_meld/preprocessed_train_splits/d...,...,15.796581,1.682826,5.095541,-3.319125,M,high,high,high,high,high (tense/bright)
955,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,1.664,12,165,neutral,"No-no, Kathy.",Joey,0.670000,M,/content/prep_meld/preprocessed_train_splits/d...,...,7.636024,0.713922,3.278688,-5.037021,M,moderate,high,low,moderate,high (tense/bright)
4049,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,1.429,5,703,surprise,What?,Monica,0.650000,F,/content/prep_meld/preprocessed_train_splits/d...,...,4.180851,0.523066,3.389831,1.612725,F,high,moderate,low,moderate,high (tense/bright)
3177,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,8.469,7,560,joy,Here we go! Here we go! Here we go! How you d...,Joey,1.089000,M,/content/prep_meld/preprocessed_train_splits/d...,...,5.368694,1.511051,0.980392,-7.475145,M,low,moderate,high,low,moderate (modal)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3679,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,1.472,0,644,surprise,What happened in Atlantic City?!,Monica,1.102000,F,/content/prep_meld/preprocessed_train_splits/d...,...,3.627708,1.208333,4.854369,-5.973757,F,moderate,moderate,moderate,high,moderate (modal)
83,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,5.440,4,15,surprise,"Oh my God, wh-what happened?",Ross,0.510000,M,/content/prep_meld/preprocessed_train_splits/d...,...,10.067408,0.686057,4.444445,-9.487512,M,moderate,high,low,high,moderate (modal)
87,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,2.240,9,15,sadness,I guess.,Charlie,0.506000,F,/content/prep_meld/preprocessed_train_splits/d...,...,3.195700,0.681412,4.545455,-8.554057,F,low,low,low,high,moderate (modal)
4055,/content/drive/MyDrive/AGORA_SER/data/meld_16k...,3.349,3,705,joy,"Yeah, I already have my bell and later on...I ...",Phoebe,2.810000,F,/content/prep_meld/preprocessed_train_splits/d...,...,10.526268,1.684088,4.779412,-3.442302,F,moderate,high,high,high,moderate (modal)


## 3. One independent question per selected cue

The model receives audio, the requested cue, and saved `effective_gender` as its
acoustic reference group, identically during training and evaluation. F/M follow
the preprocessing assignment (including its pitch-based fallback); `__GLOBAL__`
explicitly requests the pooled reference when gender could not be resolved.
Missing or unknown groups fail validation rather than being guessed here.
Speaker identity, transcript, emotion, and measured features are not supplied.
Each target is exactly
`<answer>saved label</answer>`. Missing validation targets are excluded only from
that cue's scoring denominator and recorded in `coverage.csv`.

In [15]:
PERSONA = 'You analyze acoustic properties of speech. Return only the requested answer tag.'


def cue_instruction(cue, effective_gender):
    labels = ', '.join(CUE_LABELS[cue])
    groups = {'F': 'F (female)', 'M': 'M (male)',
              '__GLOBAL__': '__GLOBAL__ (pooled reference; speaker gender unresolved)'}
    if effective_gender not in groups:
        raise ValueError(f'Invalid effective_gender: {effective_gender!r}')
    return (f'Acoustic reference group (effective speaker gender): {groups[effective_gender]}. '
            f'Listen to the audio. Classify its {CUE_DESCRIPTIONS[cue]} relative to '
            f'this reference group. Choose exactly one label: {labels}. '
            'Return only <answer>label</answer>.')


def build_examples(df, cues):
    examples, coverage = [], []
    for cue in cues:
        column = f'{cue}_cue'
        valid = df[column].notna()
        coverage.append(dict(cue=cue, total=len(df), labeled=int(valid.sum()), missing=int((~valid).sum())))
        for path, gender, label in df.loc[valid, ['audio_path', 'effective_gender', column]].itertuples(index=False, name=None):
            examples.append(dict(audio_path=path, effective_gender=gender, cue=cue, truth=label,
                                 instruction=cue_instruction(cue, gender), sft_gt=f'<answer>{label}</answer>'))
    if not examples:
        raise ValueError('No labeled cue examples')
    return pd.DataFrame(examples), pd.DataFrame(coverage)


train_examples, train_coverage = build_examples(train_set, SELECTED_CUES)
val_examples, val_coverage = build_examples(val_set, SELECTED_CUES)
if (val_coverage.labeled == 0).any():
    raise ValueError('A selected cue has no validation targets')
majority_labels = {cue: group.truth.value_counts().idxmax() for cue, group in train_examples.groupby('cue')}
display(train_examples.groupby(['cue', 'truth']).size().rename('training_examples'))
display(val_coverage)

# Refuse to mix artifacts from independent runs.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
train_examples.to_csv(OUTPUT_DIR / 'train_examples.csv', index=False)
val_examples.to_csv(OUTPUT_DIR / 'val_examples.csv', index=False)
pd.concat([train_coverage.assign(split='train'), val_coverage.assign(split='val')]).to_csv(
    OUTPUT_DIR / 'coverage.csv', index=False)
manifest = dict(task='cue_only', gender_input_column='effective_gender', model=MODEL_ID, cues=list(SELECTED_CUES), seed=SEED,
                epochs=EPOCHS, train_audio=len(train_set), val_audio=len(val_set),
                majority_labels=majority_labels, quartiles=quartiles,
                sources={str(p): hashlib.sha256(p.read_bytes()).hexdigest()
                         for p in (TRAIN_CSV, VAL_CSV, QUARTILES_JSON)})
(OUTPUT_DIR / 'experiment.json').write_text(json.dumps(manifest, indent=2))

cue              truth              
pitch_level      high                   299
                 low                    308
                 moderate               604
pitch_variation  high                   301
                 low                    303
                 moderate               607
speaking_rate    high                   294
                 low                    303
                 moderate               614
timbre           high (tense/bright)    303
                 low (muffled/soft)     302
                 moderate (modal)       606
vocal_energy     high                   302
                 low                    307
                 moderate               602
Name: training_examples, dtype: int64

,cue,total,labeled,missing
0,pitch_level,816,813,3
1,pitch_variation,816,813,3
2,vocal_energy,816,816,0
3,speaking_rate,816,813,3
4,timbre,816,813,3


3523

## 4. Qwen Thinker + QLoRA

Load the same Qwen2.5-Omni checkpoint through its supported text-output
[Thinker class](https://huggingface.co/docs/transformers/v5.20.2/en/model_doc/qwen2_5_omni).
This keeps audio input and avoids the old notebook's global Talker/forward patches.
LoRA still targets only the language decoder attention and MLP projections;
the audio encoder remains frozen. The saved adapter loads onto the Thinker class
for subsequent use (its module paths differ from the full Omni wrapper).

In [16]:
import torch
from torch.utils.data import Dataset
from transformers import (AutoProcessor, Qwen2_5OmniThinkerForConditionalGeneration,
                          BitsAndBytesConfig, Trainer, TrainingArguments, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError('4-bit QLoRA training requires a CUDA GPU in this notebook.')
set_seed(SEED)
USE_BF16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'right'
model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=compute_dtype, device_map={'': torch.cuda.current_device()},
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True))
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0, bias='none', task_type=None,
    target_modules=r'model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))'))
model.config.use_cache = False
model.print_trainable_parameters()

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniThinkerForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-7B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.alpha             | UNEXPECTED |  | 
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.beta              | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.q_proj.weight                                                     | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_out.0.weight                            | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.bias                               | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transf

trainable params: 40,370,176 || all params: 8,972,184,064 || trainable%: 0.4499


## 5. Audio batches and assistant-only loss

Require the preprocessing audio contract (16 kHz mono, finite, nonempty).
Mask the prompt using its actual multimodal token prefix, and mask padding using
the attention mask so end-of-turn tokens remain supervised. Fail on a template
boundary mismatch instead of accidentally training on the prompt.

In [17]:
def load_audio(path):
    audio, sr = sf.read(path, dtype='float32')
    if sr != 16000 or audio.ndim != 1 or not len(audio) or not np.isfinite(audio).all():
        raise ValueError(f'Expected finite, nonempty 16 kHz mono audio: {path}')
    return audio


def messages_for(item):
    return [{'role': 'system', 'content': PERSONA},
            {'role': 'user', 'content': [{'type': 'audio', 'audio': item['audio_path']},
                                        {'type': 'text', 'text': item['instruction']}]}]


class CueDataset(Dataset):
    def __init__(self, df):
        self.records = df.to_dict('records')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


def mask_answer_labels(full, prompt):
    labels = full['input_ids'].clone()
    for i in range(len(labels)):
        prompt_ids = prompt['input_ids'][i][prompt['attention_mask'][i].bool()]
        full_positions = full['attention_mask'][i].nonzero(as_tuple=True)[0]
        full_ids = full['input_ids'][i][full_positions]
        n = len(prompt_ids)
        if n >= len(full_ids) or not (full_ids[:n] == prompt_ids).all():
            raise ValueError('Chat template prompt is not an exact prefix of the target sequence')
        labels[i, full_positions[:n]] = -100
    labels[full['attention_mask'] == 0] = -100
    return labels


class CueCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, items):
        audios = [load_audio(item['audio_path']) for item in items]
        prompts, targets = [], []
        for item in items:
            messages = messages_for(item)
            prompts.append(self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            targets.append(self.processor.apply_chat_template(
                messages + [{'role': 'assistant', 'content': item['sft_gt']}],
                tokenize=False, add_generation_prompt=False))
        kwargs = dict(audio=audios, sampling_rate=16000, padding=True, return_tensors='pt')
        full = self.processor(text=targets, **kwargs)
        prompt = self.processor(text=prompts, **kwargs)
        full['labels'] = mask_answer_labels(full, prompt)
        return full

## 6. Generated-label evaluation

Greedy generation uses the audio and question including the saved reference group. Invalid or repeated answers
count as errors. Report accuracy, macro recall (UAR), macro F1, per-class recall,
and prediction counts, including invalid outputs. Compare with the **training**
majority-label baseline on the same validation examples. Teacher-forced loss is
a separate optimization diagnostic, not the learnability score.

A progress bar shows completed examples, elapsed time, estimated time remaining,
and processing speed for each before/after evaluation. The estimate starts after
the first batch completes and updates as generation proceeds.


In [ ]:
from tqdm.auto import tqdm

def parse_answer(text, cue):
    match = re.fullmatch(r'\s*<answer>\s*(.*?)\s*</answer>\s*', text, flags=re.DOTALL | re.IGNORECASE)
    label = match.group(1).strip().lower() if match else 'invalid'
    return label if label in CUE_LABELS[cue] else 'invalid'


def cue_metrics(records):
    report = {}
    for cue, group in records.groupby('cue'):
        labels = CUE_LABELS[cue]
        precision, recall, f1, support = precision_recall_fscore_support(
            group.truth, group.prediction, labels=list(labels), zero_division=0)
        report[cue] = dict(
            n=len(group), accuracy=float(accuracy_score(group.truth, group.prediction)),
            macro_recall=float(recall.mean()), macro_f1=float(f1.mean()),
            invalid_rate=float((group.prediction == 'invalid').mean()),
            prediction_counts={str(k): int(v) for k, v in group.prediction.value_counts().items()},
            per_class={label: dict(precision=float(precision[i]), recall=float(recall[i]),
                                  f1=float(f1[i]), support=int(support[i]))
                       for i, label in enumerate(labels)})
    return report


def evaluate_generated(model, processor, examples, batch_size=2, description='Cue validation'):
    was_training = model.training
    padding_side = processor.tokenizer.padding_side
    cache = model.config.use_cache
    records = []
    try:
        model.eval()
        model.config.use_cache = True
        processor.tokenizer.padding_side = 'left'
        with torch.inference_mode(), tqdm(
                total=len(examples), desc=description, unit='example',
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed} elapsed, {remaining} remaining, {rate_fmt}]'
        ) as progress:
            for start in range(0, len(examples), batch_size):
                items = examples.iloc[start:start + batch_size].to_dict('records')
                prompts = [processor.apply_chat_template(messages_for(item), tokenize=False,
                                                         add_generation_prompt=True) for item in items]
                inputs = processor(text=prompts, audio=[load_audio(item['audio_path']) for item in items],
                                   sampling_rate=16000, padding=True, return_tensors='pt').to(model.device)
                generated = model.generate(**inputs, do_sample=False, max_new_tokens=48,
                                           pad_token_id=processor.tokenizer.pad_token_id)
                texts = processor.tokenizer.batch_decode(
                    generated[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                for item, text in zip(items, texts):
                    records.append(dict(audio_path=item['audio_path'], effective_gender=item['effective_gender'],
                                        cue=item['cue'], truth=item['truth'],
                                        prediction=parse_answer(text, item['cue']), completion=text))
                progress.update(len(items))
    finally:
        processor.tokenizer.padding_side = padding_side
        model.config.use_cache = cache
        model.train(was_training)
    records = pd.DataFrame(records)
    return records, cue_metrics(records)


def save_evaluation(stage):
    records, metrics = evaluate_generated(model, processor, val_examples, EVAL_BATCH_SIZE,
                                          description=f'Validation ({stage} training)')
    records.to_csv(OUTPUT_DIR / f'{stage}_predictions.csv', index=False)
    (OUTPUT_DIR / f'{stage}_metrics.json').write_text(json.dumps(metrics, indent=2))
    return metrics


baseline = val_examples.assign(prediction=val_examples.cue.map(majority_labels))
baseline_metrics = cue_metrics(baseline)
(OUTPUT_DIR / 'majority_baseline.json').write_text(json.dumps(baseline_metrics, indent=2))
before_metrics = save_evaluation('before') if EVALUATE_BEFORE else None

## 7. Train, evaluate the saved val split, and save

Defaults preserve the original one-epoch, batch size 2, accumulation 4, learning
rate 1e-4, cosine schedule, and weight decay 0.01. All selected cues share one
adapter. No validation sampling or training-pool holdout is performed.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=EPOCHS,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=1e-4, lr_scheduler_type='cosine',
    weight_decay=0.01, max_grad_norm=1.0, logging_steps=5,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    bf16=USE_BF16, fp16=not USE_BF16, gradient_checkpointing=True,
    dataloader_pin_memory=False, remove_unused_columns=False,
    prediction_loss_only=True, label_names=['labels'], report_to='none', seed=SEED,
)
trainer = Trainer(model=model, args=training_args,
                  train_dataset=CueDataset(train_examples), eval_dataset=CueDataset(val_examples),
                  data_collator=CueCollator(processor))
trainer.train()
loss_metrics = trainer.evaluate()
(OUTPUT_DIR / 'validation_loss.json').write_text(json.dumps(loss_metrics, indent=2))
trainer.save_model(str(OUTPUT_DIR / 'final_sft_adapter'))
processor.save_pretrained(OUTPUT_DIR / 'final_sft_adapter')
after_metrics = save_evaluation('after')

summary = []
for stage, metrics in [('train_majority', baseline_metrics), ('before', before_metrics), ('after', after_metrics)]:
    if metrics is not None:
        for cue, values in metrics.items():
            summary.append(dict(stage=stage, cue=cue, **{k: values[k] for k in
                ('n', 'accuracy', 'macro_recall', 'macro_f1', 'invalid_rate')}))
summary = pd.DataFrame(summary)
summary.to_csv(OUTPUT_DIR / 'validation_summary.csv', index=False)
display(summary)
print(f'Adapter, provenance, coverage, and validation results saved to {OUTPUT_DIR}')

Read the after-training scores alongside the base-model and training-majority
baselines. High accuracy with poor low/high recall suggests majority collapse.
Inspect per-class recall and prediction counts in `after_metrics.json` and raw
completions in `after_predictions.csv`. Improvement on saved validation labels
supports learnability of this cue scheme; it does not establish causal acoustic
reasoning or downstream GRPO/emotion performance. Validation coverage is relative
to the cleaned manifest, not the unfiltered original MELD dev set.